In [1]:
from google.colab import auth
auth.authenticate_user()

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [13]:
%cd /content
!git clone https://github.com/UrbinaDan/PaperTheater_ai_Pipeline.git
%cd PaperTheater_ai_Pipeline

/content
fatal: destination path 'PaperTheater_ai_Pipeline' already exists and is not an empty directory.
/content/PaperTheater_ai_Pipeline


In [14]:
!pip install -q \
  "numpy>=2.0" \
  "scipy" \
  "matplotlib" \
  "opencv-python-headless" \
  "pillow" \
  "shapely" \
  "svgwrite" \
  "cairosvg" \
  "tqdm" \
  "networkx" \
  "pyyaml" \
  "requests" \
  "openai" \
  "accelerate" \
  "bitsandbytes" \
  "einops" \
  "sentencepiece" \
  "transformers==4.49.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 108.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 4.29.0 requires numpy~=1.0, but you have numpy 2.4.3 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.3 which is incompatible.
yfinance 0.2.66 requires websockets>=13.0, but you have websockets 11.0.3 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.3 which is incompatible.


In [15]:
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA: True
GPU: Tesla T4


In [ ]:
%cd /content
!git clone https://github.com/facebookresearch/sam2.git
%cd /content/sam2
!pip install -e .

/content
fatal: destination path 'sam2' already exists and is not an empty directory.
/content/sam2
Obtaining file:///content/sam2


In [ ]:
%cd /content/sam2
!mkdir -p checkpoints
!wget -q -P checkpoints https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_small.pt
!wget -q -P checkpoints https://raw.githubusercontent.com/facebookresearch/sam2/main/sam2_configs/sam2_hiera_s.yaml

In [ ]:
%cd /content
!git clone https://github.com/DepthAnything/Depth-Anything-V2.git
%cd /content/Depth-Anything-V2
!pip install -q -r requirements.txt

In [ ]:
%cd /content/Depth-Anything-V2
!mkdir -p checkpoints
!wget -q -P checkpoints https://huggingface.co/depth-anything/Depth-Anything-V2-Small/resolve/main/depth_anything_v2_vits.pth

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI key: ")

In [ ]:
%cd /content/PaperTheater_ai_Pipeline
import sys
sys.path.append("/content/PaperTheater_ai_Pipeline")

In [ ]:
!pip uninstall -y transformers tokenizers
!pip install -q "transformers==4.49.0" "tokenizers==0.21.4"

TESTS


In [ ]:
from src.depth_anything_runner import DepthRunner
depth_runner = DepthRunner()
print("Depth model loaded")

In [ ]:
from src.sam2_segmenter import SAM2Segmenter
segmenter = SAM2Segmenter()
print("SAM2 loaded")

In [ ]:
from src.config import Paths, PipelineConfig
from src.io_utils import ensure_dirs, load_image
from src.florence_parser import FlorenceParser
from src.scene_builder import parse_florence_boxes

paths = Paths()
cfg = PipelineConfig()
ensure_dirs(paths)
print("Setup imports loaded")

**Occlusion and completion**
Step 2

In [ ]:
%cd /content/PaperTheater_ai_Pipeline

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from src.config import Paths, PipelineConfig
from src.io_utils import ensure_dirs, load_image, save_mask, save_json
from src.florence_parser import FlorenceParser
from src.sam2_segmenter import SAM2Segmenter
from src.depth_anything_runner import DepthRunner
from src.scene_builder import parse_florence_boxes, assign_layers
from src.occlusion_heuristic import heuristic_complete
from src.occlusion_amodal import amodal_experimental
from src.mask_cleanup import cleanup_mask


In [ ]:
paths = Paths()
cfg = PipelineConfig()
ensure_dirs(paths)

In [ ]:
image_path = "/content/PaperTheater_ai_Pipeline/data/input/scene_1.jpg"
image = load_image(image_path, max_side=cfg.image_max_side)

plt.figure(figsize=(8, 8))
plt.imshow(image)
plt.axis("off")
plt.show()

In [ ]:
%cd /content/PaperTheater_ai_Pipeline

import importlib
import src.florence_parser
importlib.reload(src.florence_parser)
import src.scene_builder
importlib.reload(src.scene_builder)

from src.scene_builder import parse_florence_boxes

from src.florence_parser import FlorenceParser

In [ ]:
florence = FlorenceParser(cfg.florence_model)

caption = florence.get_dense_caption(image)
caption

In [ ]:
det = florence.get_open_vocab_detection(
    image,
    "a tree, a temple, a pagoda, a building, a bridge, a mountain, the sky, bushes, foreground plants"
)
det

In [ ]:
boxes = parse_florence_boxes(det)
boxes[:5]

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(image)

for b in boxes:
    x1, y1, x2, y2 = b["bbox"]
    rect = patches.Rectangle(
        (x1, y1), x2 - x1, y2 - y1,
        linewidth=2, edgecolor='red', facecolor='none'
    )
    ax.add_patch(rect)
    ax.text(x1, y1 - 5, b["label"], color='yellow', fontsize=10, backgroundcolor='black')

ax.axis("off")
plt.show()

In [ ]:
segmenter = SAM2Segmenter(cfg.sam2_config, cfg.sam2_checkpoint)
segmented = segmenter.segment_boxes(image, boxes)

len(segmented), segmented[0].keys()

In [ ]:
sample = segmented[0]
plt.figure(figsize=(8, 8))
plt.imshow(image)
plt.imshow(sample["mask"], alpha=0.4)
plt.title(sample["label"])
plt.axis("off")
plt.show()

In [ ]:
depth_runner = DepthRunner(cfg.depth_encoder)
depth = depth_runner.infer(image)

plt.figure(figsize=(8, 8))
plt.imshow(depth, cmap="plasma")
plt.axis("off")
plt.title("Depth map")
plt.show()

In [ ]:
layer_assignments = assign_layers(segmented, depth, cfg.target_num_layers)
layer_assignments

In [ ]:
heuristic_results = []

for i, obj in enumerate(segmented):
    completed = heuristic_complete(obj["mask"], obj["label"])
    cleaned = cleanup_mask(completed, cfg.min_component_area, cfg.smooth_kernel)

    out_path = paths.completed_dir / f"heuristic_{i:03d}.png"
    save_mask(out_path, cleaned)

    heuristic_results.append({
        "index": i,
        "label": obj["label"],
        "bbox": obj["bbox"],
        "layer": layer_assignments[i],
        "mask_path": str(out_path)
    })

heuristic_results[:3]

In [ ]:
from src.occlusion_openai import openai_edit
from PIL import Image

In [ ]:
openai_results = []

for i, obj in enumerate(segmented):
    guess = heuristic_complete(obj["mask"], obj["label"])
    cleaned_mask = cleanup_mask(guess, cfg.min_component_area, cfg.smooth_kernel)

    # use this only on structured labels
    label = obj["label"].lower()
    structured = any(x in label for x in ["temple", "pagoda", "building", "house", "bridge", "roof"])

    if structured:
        edited = openai_edit(image, cleaned_mask, obj["label"], cfg.openai_model)
        out_img = paths.completed_dir / f"openai_edit_{i:03d}.png"
        Image.fromarray(edited).save(out_img)

    out_mask = paths.completed_dir / f"openai_mask_{i:03d}.png"
    save_mask(out_mask, cleaned_mask)

    openai_results.append({
        "index": i,
        "label": obj["label"],
        "bbox": obj["bbox"],
        "layer": layer_assignments[i],
        "mask_path": str(out_mask)
    })

In [ ]:
amodal_results = []

for i, obj in enumerate(segmented):
    completed = amodal_experimental(obj["mask"], obj["label"])
    cleaned = cleanup_mask(completed, cfg.min_component_area, cfg.smooth_kernel)

    out_path = paths.completed_dir / f"amodal_{i:03d}.png"
    save_mask(out_path, cleaned)

    amodal_results.append({
        "index": i,
        "label": obj["label"],
        "bbox": obj["bbox"],
        "layer": layer_assignments[i],
        "mask_path": str(out_path)
    })



## --- SECTION 3
VECTOR EXPORT




In [ ]:
%cd /content/paper-theater-ai

from src.config import Paths, PipelineConfig
from src.io_utils import load_mask
from src.vectorize import mask_to_polygons
from src.fabrication import merge_polygons, thicken_fragile_parts, remove_tiny_parts
from src.export_svg import save_svg
import numpy as np

In [ ]:
def export_branch(branch_results, branch_name, image_shape):
    paths = Paths()
    cfg = PipelineConfig()

    h, w = image_shape[:2]

    for layer_idx in range(cfg.target_num_layers):
        masks = []
        for item in branch_results:
            if item["layer"] == layer_idx:
                masks.append(load_mask(item["mask_path"]))

        if not masks:
            continue

        merged = np.zeros_like(masks[0], dtype=np.uint8)
        for m in masks:
            merged = ((merged > 0) | (m > 0)).astype(np.uint8)

        polys = mask_to_polygons(merged, cfg.simplify_tolerance)
        geom = merge_polygons(polys)
        geom = remove_tiny_parts(geom, 100)
        geom = thicken_fragile_parts(geom, 2)

        out_path = paths.layers_svg_dir / f"{branch_name}_layer_{layer_idx}.svg"
        save_svg(geom, out_path, w, h)
        print("saved", out_path)

In [ ]:
from PIL import Image
from pathlib import Path
import json

image_path = Paths().input_dir / "scene.png"
img = Image.open(image_path).convert("RGB")
img_np = np.array(img)

# reuse branch results from earlier notebook or load from saved json if you stored them
export_branch(heuristic_results, "heuristic", img_np.shape)
export_branch(amodal_results, "amodal", img_np.shape)
export_branch(openai_results, "openai", img_np.shape)

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image

paths = Paths()

for p in sorted(paths.layers_svg_dir.glob("*.svg")):
    print(p.name)

##

---SECTION 4
Compare Results

